In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.externals import joblib
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_validate, cross_val_predict
from sklearn.preprocessing import MultiLabelBinarizer
from preprocessing import TagGrouper, MultilabelFeatureSelector
from plotting import plot_multilabel_classification_report, plot_multilabel_precision_recall, plot_multilabel_roc
from persistence import load_all_classifiers
from baseline import BASELINE_FEATURE_COLUMNS

In [ ]:
# Load classifiers
classifiers = load_all_classifiers('models-baseline-0c25f9f')

---
## Import Dataset

In [ ]:
# Import dataset
df = pd.read_csv('./data/features-0c25f9f.csv', index_col='revision_id')
tag_df = pd.read_csv('./data/tag.csv', index_col='id')
df.head()

In [ ]:
# Tag distribution
sns.distplot(df['tag_id'], kde=False)
plt.show()

---
## Preprocessing

In [ ]:
def preprocessing(frame, features):
    feature_columns = None
    
    if features == 'baseline':
        feature_columns = BASELINE_FEATURE_COLUMNS

    preprocess_pipeline = Pipeline([
        ('feature_selector', MultilabelFeatureSelector(feature_columns)),
        ('tag_grouper', TagGrouper())
    ])

    return preprocess_pipeline.transform(frame)\
        .fillna(0)


def flatten_multilabel_predict(y_score):
    result = []
    for i in range(len(y_score[0])):
        row = []
        for l in range(len(y_score)):
            row.append(y_score[l][i]) 
        result.append(row)
    return np.array(result)


def flatten_multilabel_predict_proba(y_score):
    result = []
    for i in range(len(y_score[0])):
        row = []
        for l in range(len(y_score)):
            row.append(y_score[l][i][1]) 
        result.append(row)
    return np.array(result)


tags = [1, 2, 3, 5, 6, 9, 11, 12]
tag_names = ['Blanking', 'Nonsense', 'QualityIssue', 'FalseFact', 'Syntax', 'EditWar', 'Intention', 'Constructive']
print('Selected tags', tags)

In [ ]:
processed_df = preprocessing(df, 'baseline')
processed_df.head()

In [ ]:
# X, y split
X = processed_df.drop(['all_tags'], axis=1)
y = MultiLabelBinarizer().fit_transform(processed_df['all_tags'])

In [ ]:
# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
print('X train', X_train.shape)
print('y train', y_train.shape)
print('X test', X_test.shape)
print('y test', y_test.shape)

In [ ]:
estimators = [(str(tag_id), model['clf']) for tag_id, model in classifiers.items()]
estimators = sorted(estimators, key=lambda e: int(e[0]))
y_test_predict_unprocessed = []
y_test_predict_proba_unprocessed = []

# Train classifiers and make predicitions per tag
for i, e in enumerate(estimators):
    clf = e[1]
    # Retrain classifier on tag data
    y_train_tag = y_train[:, i]
    clf.fit(X_train, y_train_tag)
    
    # Predict tag for test
    y_test_predict_unprocessed.append(clf.predict(X_test))
    y_test_predict_proba_unprocessed.append(clf.predict_proba(X_test))

In [ ]:
y_test_predict = flatten_multilabel_predict(y_test_predict_unprocessed)
y_test_predict_proba = flatten_multilabel_predict_proba(y_test_predict_proba_unprocessed)

In [ ]:
plot_multilabel_classification_report(y_test, y_test_predict, tag_names)

In [ ]:
y_test_predict_proba

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, average_precision_score, auc, roc_curve
from sklearn.feature_selection import mutual_info_classif
from scipy import interp


def plot_multilabel_precision_recall2(Y_true, Y_predict_proba, tags, tag_names, output_path=''):
    plt.figure(figsize=(8, 8))
    precision = dict()
    recall = dict()
    average_precision = dict()

    for i in range(0, len(Y_true[0])):
        precision[i], recall[i], _ = precision_recall_curve(Y_true[:, i], Y_predict_proba[:, i])
        average_precision[i] = average_precision_score(Y_true[:, i], Y_predict_proba[:, i])

    f_scores = np.linspace(0.2, 0.8, num=4)
    lines = []
    labels = []
    for f_score in f_scores:
        x = np.linspace(0.01, 1)
        y = f_score * x / (2 * x - f_score)
        l, = plt.plot(x[y >= 0], y[y >= 0], color='gray', alpha=0.2)
        plt.annotate('f1={0:0.1f}'.format(f_score), xy=(0.9, y[45] + 0.03))

    lines.append(l)
    labels.append('iso-f1 curves')
    colors = plt.get_cmap('tab20').colors

    for i in range(len(tag_names)):
        l, = plt.plot(recall[i], precision[i], lw=2, color = colors[tags[i] - 1])
        lines.append(l)
        labels.append('{0} (AUC = {1:0.2f})'
                    ''.format(tag_names[i], average_precision[i]))

    fig = plt.gcf()
    fig.subplots_adjust(bottom=0.25)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Recall')
    plt.ylabel('Precision')

    plt.legend(lines, labels, loc=9, bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False)
    plt.savefig(output_path, bbox_inches='tight')
    plt.show()

plot_multilabel_precision_recall2(y_test, y_test_predict_proba, tags, tag_names, 'figures/multilabel/multilabel_baseline_precision_recall.pdf')

In [ ]:
plot_multilabel_roc(y_test, y_test_predict_proba, tags, tag_names, 'figures/multilabel/multilabel_baseline_roc.pdf')